# Segundo Sprint — Proyecto NLP
## Análisis de Sentimiento en Tweets (Sentiment140)


## 0. Instalación de dependencias e imports

In [ ]:
# Librerías que no vienen preinstaladas en Colab
!pip install -q kagglehub wordcloud umap-learn keybert


In [ ]:
import os
import re
import random
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from gensim.models import Word2Vec
from textblob import TextBlob

nltk.download("stopwords")

random.seed(42)
np.random.seed(42)


In [ ]:
# Parámetros generales del notebook
USE_SAMPLE = True        # True = corrida rápida de validación / False = corrida final con el dataset completo
SAMPLE_SIZE = 50000      # tamaño de la muestra cuando USE_SAMPLE = True
RANDOM_STATE = 42


## 1. Carga de datos

Descargamos el dataset Sentiment140 completo (1.6M de tweets de entrenamiento + 498 tweets de test manual con las 3 clases 0/2/4). Ver sección 1 y 3 de la guía para el detalle de columnas y la razón de usar `encoding="ISO-8859-1"`.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("kazanova/sentiment140")
print("Dataset descargado en:", path)

cols = ["target", "ids", "date", "flag", "user", "text"]

df = pd.read_csv(f"{path}/training.1600000.processed.noemoticon.csv",
                  encoding="ISO-8859-1", names=cols)

df_test_manual = pd.read_csv(f"{path}/testdata.manual.2009.06.14.csv",
                              encoding="ISO-8859-1", names=cols)

print("Train:", df.shape, "| Test manual:", df_test_manual.shape)


In [ ]:
if USE_SAMPLE:
    df = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"USE_SAMPLE=True -> trabajando con una muestra de {len(df)} tweets (para validar el pipeline).")
else:
    print(f"USE_SAMPLE=False -> trabajando con el dataset COMPLETO ({len(df)} tweets), como pide el enunciado.")


## 2. Análisis exploratorio (EDA)

### 2.1 Chequeos básicos

In [ ]:
df.isna().sum()


In [ ]:
df.duplicated().sum()


In [ ]:
df["target"].value_counts()   # esperado: solo 0 y 4 en el set de train


In [ ]:
mapping = {0: "negative", 2: "neutral", 4: "positive"}
df["sentiment"] = df["target"].map(mapping)
df_test_manual["sentiment"] = df_test_manual["target"].map(mapping)

df["sentiment"].value_counts(normalize=True) * 100


In [ ]:
df["text"].str.len().describe()


In [ ]:
df[df.sentiment == "positive"].text.sample(5, random_state=RANDOM_STATE)


In [ ]:
df[df.sentiment == "negative"].text.sample(5, random_state=RANDOM_STATE)


In [ ]:
plt.figure(figsize=(10, 5))
df["text"].str.len().hist(bins=50)
plt.xlabel("Longitud del tweet (caracteres)")
plt.ylabel("Cantidad de tweets")
plt.title("Distribución de longitud de tweets")
plt.show()


### 2.2 Palabras más frecuentes y distintivas por clase

No alcanza con mirar ejemplos sueltos: conviene ver qué palabras aparecen con más frecuencia (y, sobre todo, con más *proporción relativa*) en tweets positivos vs. negativos. Usamos una tokenización liviana propia de esta exploración (no la pipeline oficial de limpieza, que se define recién en la sección 3), sacando stopwords para que no dominen los conteos.

In [ ]:
from collections import Counter

stop_words_eda = set(stopwords.words("english"))
token_re_eda = re.compile(r"[a-zA-Z]+")

def tokens_eda(texto):
    return [w for w in token_re_eda.findall(texto.lower()) if w not in stop_words_eda and len(w) > 1]

freq_pos = Counter(w for tokens in df[df.target == 4]["text"].apply(tokens_eda) for w in tokens)
freq_neg = Counter(w for tokens in df[df.target == 0]["text"].apply(tokens_eda) for w in tokens)

print("Top 15 palabras más frecuentes en tweets POSITIVOS:", freq_pos.most_common(15))
print("\nTop 15 palabras más frecuentes en tweets NEGATIVOS:", freq_neg.most_common(15))


In [ ]:
# No sólo frecuencia bruta: proporción dentro de cada clase, y un ratio para ver qué palabras
# son realmente DISTINTIVAS de cada sentimiento (no sólo frecuentes en general).
def tabla_frecuencias_por_clase(freq_pos, freq_neg, min_count=30):
    vocab = set(freq_pos) | set(freq_neg)
    total_pos, total_neg = sum(freq_pos.values()), sum(freq_neg.values())
    filas = []
    for w in vocab:
        cp, cn = freq_pos.get(w, 0), freq_neg.get(w, 0)
        if cp + cn < min_count:
            continue
        prop_pos, prop_neg = cp / total_pos, cn / total_neg
        ratio = (prop_pos + 1e-6) / (prop_neg + 1e-6)
        filas.append({"palabra": w, "freq_pos": cp, "freq_neg": cn,
                       "prop_pos_%": prop_pos * 100, "prop_neg_%": prop_neg * 100,
                       "ratio_pos_neg": ratio})
    return pd.DataFrame(filas)

tabla_frecuencias = tabla_frecuencias_por_clase(freq_pos, freq_neg)

print("Palabras más DISTINTIVAS de tweets POSITIVOS (mayor ratio pos/neg):")
tabla_frecuencias.sort_values("ratio_pos_neg", ascending=False).head(15)


In [ ]:
print("Palabras más DISTINTIVAS de tweets NEGATIVOS (menor ratio pos/neg):")
tabla_frecuencias.sort_values("ratio_pos_neg", ascending=True).head(15)


In [ ]:
top_pos = tabla_frecuencias.sort_values("ratio_pos_neg", ascending=False).head(15)
top_neg = tabla_frecuencias.sort_values("ratio_pos_neg", ascending=True).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(top_pos["palabra"][::-1], top_pos["ratio_pos_neg"][::-1], color="seagreen")
axes[0].set_title("Palabras más distintivas de tweets POSITIVOS")
axes[0].set_xlabel("Ratio proporción positivo / negativo")

axes[1].barh(top_neg["palabra"][::-1], (1 / top_neg["ratio_pos_neg"])[::-1], color="indianred")
axes[1].set_title("Palabras más distintivas de tweets NEGATIVOS")
axes[1].set_xlabel("Ratio proporción negativo / positivo")

plt.tight_layout()
plt.show()


### 2.3 Nuevos atributos derivados del texto y la fecha

Creamos atributos adicionales a partir del texto crudo y de la columna `date`, para después compararlos contra el sentimiento.

In [ ]:
url_re_eda = re.compile(r"https?://\S+|www\.\S+")
hashtag_re_eda = re.compile(r"#\w+")
mention_re_eda = re.compile(r"@\w+")

df["text_length"] = df["text"].str.len()
df["url_count"] = df["text"].apply(lambda t: len(url_re_eda.findall(t)))
df["hashtag_count"] = df["text"].apply(lambda t: len(hashtag_re_eda.findall(t)))
df["mention_count"] = df["text"].apply(lambda t: len(mention_re_eda.findall(t)))

df[["text_length", "url_count", "hashtag_count", "mention_count"]].describe()


In [ ]:
# La columna date tiene el formato "Mon Apr 06 22:19:45 PDT 2009": la parseamos a mano
# (evita problemas de zona horaria "PDT"/"PST" que strptime no siempre reconoce bien).
def parsear_fecha(date_str):
    try:
        partes = date_str.split()
        weekday = partes[0]                    # ej. "Mon"
        hour = int(partes[3].split(":")[0])    # HH de HH:MM:SS
        return pd.Series({"weekday": weekday, "hour": hour})
    except Exception:
        return pd.Series({"weekday": np.nan, "hour": np.nan})

fecha_features = df["date"].apply(parsear_fecha)
df["weekday"] = fecha_features["weekday"]
df["hour"] = fecha_features["hour"]

df[["date", "weekday", "hour"]].head()


### 2.4 Atributos vs. variable target (sentimiento)

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="sentiment", y="text_length", data=df, order=["negative", "positive"])
plt.title("Longitud del tweet según sentimiento")
plt.show()


In [ ]:
df.groupby("sentiment")[["url_count", "hashtag_count", "mention_count"]].mean()


In [ ]:
df.groupby("sentiment")[["url_count", "hashtag_count", "mention_count"]].mean().plot(kind="bar", figsize=(8, 5))
plt.title("Promedio de URLs, hashtags y menciones por sentimiento")
plt.ylabel("Promedio por tweet")
plt.xticks(rotation=0)
plt.show()


In [ ]:
orden_dias = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
tabla_dia_sentimiento = pd.crosstab(df["weekday"], df["sentiment"], normalize="index").reindex(orden_dias) * 100
tabla_dia_sentimiento.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.title("Proporción de sentimiento por día de la semana")
plt.ylabel("% de tweets")
plt.xticks(rotation=0)
plt.show()


In [ ]:
tabla_hora_sentimiento = pd.crosstab(df["hour"], df["sentiment"], normalize="index") * 100
tabla_hora_sentimiento.plot(figsize=(10, 5), marker="o")
plt.title("Proporción de sentimiento por hora del día")
plt.xlabel("Hora")
plt.ylabel("% de tweets")
plt.show()


**Nota:** si `url_count`, `hashtag_count` y `mention_count` dan valores muy bajos y parecidos entre clases, es un resultado válido en sí mismo (indica que esos atributos aportan poca señal para distinguir sentimiento en este dataset) — no hace falta que "den distinto" para que valga la pena reportarlo. Los patrones por hora/día pueden reflejar tanto comportamiento real de los usuarios como artefactos de cómo se recolectó el dataset en 2009; vale la pena mencionarlo como limitación al sacar conclusiones de estos gráficos.

## 3. Preprocesamiento de texto

Limpieza rápida con regex + stopwords de NLTK (ver sección 5 de la guía para por qué se elige este camino en vez de spaCy token por token sobre 1.6M de filas).

In [ ]:
stop_words = set(stopwords.words("english"))
url_re = re.compile(r"https?://\S+|www\.\S+")
mention_re = re.compile(r"@\w+")
non_alpha_re = re.compile(r"[^a-zA-Z\s]")

def limpieza_rapida(texto):
    texto = texto.lower()
    texto = url_re.sub(" ", texto)
    texto = mention_re.sub(" ", texto)          # @usuario
    texto = texto.replace("#", " ")             # dejamos la palabra, sacamos el símbolo
    texto = non_alpha_re.sub(" ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    tokens = [t for t in texto.split() if t not in stop_words and len(t) > 1]
    return " ".join(tokens)


In [ ]:
df["text_clean"] = df["text"].apply(limpieza_rapida)
df_test_manual["text_clean"] = df_test_manual["text"].apply(limpieza_rapida)

df[["text", "text_clean"]].head()


**(Opcional)** Lematización de mayor calidad con spaCy, sólo recomendable si tenés tiempo/GPU o si la corrés sobre una muestra (ver Nota de la sección 5 de la guía):

In [ ]:
RUN_LEMMATIZATION = False  # poné True si querés probar esta alternativa

if RUN_LEMMATIZATION:
    import spacy
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

    def lematizar_batch(textos, batch_size=1000, n_process=2):
        resultados = []
        for doc in nlp.pipe(textos, batch_size=batch_size, n_process=n_process):
            resultados.append(" ".join(tok.lemma_ for tok in doc if not tok.is_stop))
        return resultados

    df["text_lemma"] = lematizar_batch(df["text_clean"].tolist())
    df[["text_clean", "text_lemma"]].head()


## 4. División de datos

Split interno estratificado (90/10) sobre el train, más el test manual completo como evaluación final "real" (incluye la clase neutral que el train no tiene). Ver sección 6 de la guía.

In [ ]:
X = df["text_clean"]
y = df["target"]   # 0 / 4

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=RANDOM_STATE, stratify=y
)

X_test_manual = df_test_manual["text_clean"]
y_test_manual = df_test_manual["target"]  # incluye 0, 2, 4

ENTRENADO_CON = f"{len(X_train)} tweets"
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test manual:", X_test_manual.shape)


In [ ]:
# Estructura para ir guardando los resultados de cada modelo en LOS TRES conjuntos
# (train, validación y test manual) y poder compararlos al final (sección 9).
resultados = []
predicciones_test_manual = {}

def _metricas(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    return acc, f1_macro

def registrar_resultado(nombre, entrenado_con,
                         y_train, y_train_pred,
                         y_val, y_val_pred,
                         y_test, y_test_pred):
    acc_train, f1_train = _metricas(y_train, y_train_pred)
    acc_val, f1_val = _metricas(y_val, y_val_pred)
    acc_test, f1_test = _metricas(y_test, y_test_pred)

    print(f"\n========== {nombre} ==========")
    print(f"--- Train (n={len(y_train)}) ---")
    print(classification_report(y_train, y_train_pred, zero_division=0))
    print(f"--- Validación interna (n={len(y_val)}) ---")
    print(classification_report(y_val, y_val_pred, zero_division=0))
    print(f"--- Test manual (n={len(y_test)}, incluye clase neutral) ---")
    print(classification_report(y_test, y_test_pred, zero_division=0))

    resultados.append({
        "modelo": nombre, "entrenado_con": entrenado_con,
        "accuracy_train": acc_train, "f1_macro_train": f1_train,
        "accuracy_val": acc_val, "f1_macro_val": f1_val,
        "accuracy_test_manual": acc_test, "f1_macro_test_manual": f1_test,
    })
    predicciones_test_manual[nombre] = np.asarray(y_test_pred)


## 5. Baseline: modelo pre-entrenado (TextBlob)

In [ ]:
def polaridad_textblob(texto):
    return TextBlob(texto).sentiment.polarity

def a_clase_binaria(p):
    return 4 if p >= 0 else 0

def a_clase_ternaria(p, umbral=0.05):
    if p > umbral:
        return 4
    elif p < -umbral:
        return 0
    else:
        return 2


In [ ]:
# TextBlob no se entrena, así que no hace falta correrlo sobre las 1.6M filas de train.
# Para evaluarlo igual "en los 3 conjuntos" (train/val/test) usamos una muestra representativa
# de train y de validación, y el test manual COMPLETO (que es el que realmente importa comparar).
N_MUESTRA_TB = 5000

muestra_train_tb = df.sample(min(N_MUESTRA_TB, len(df)), random_state=RANDOM_STATE)
tb_pred_train = muestra_train_tb["text_clean"].apply(polaridad_textblob).apply(a_clase_binaria)

muestra_val_tb = X_val.sample(min(N_MUESTRA_TB, len(X_val)), random_state=RANDOM_STATE)
y_val_muestra_tb = y_val.loc[muestra_val_tb.index]
tb_pred_val = muestra_val_tb.apply(polaridad_textblob).apply(a_clase_binaria)


In [ ]:
# Evaluación "oficial" sobre el test manual (para la tabla comparativa de la sección 9)
tb_pred_test = df_test_manual["text_clean"].apply(polaridad_textblob).apply(a_clase_ternaria)

registrar_resultado("TextBlob (pre-entrenado)", "— (no se entrena; muestra de train/val)",
                     muestra_train_tb["target"], tb_pred_train,
                     y_val_muestra_tb, tb_pred_val,
                     y_test_manual, tb_pred_test)


## 6. Modelo 1: Bag of Words + Naive Bayes

In [ ]:
vectorizer_bow = CountVectorizer(max_features=50000, ngram_range=(1, 1))
X_train_bow = vectorizer_bow.fit_transform(X_train)
X_val_bow = vectorizer_bow.transform(X_val)
X_test_manual_bow = vectorizer_bow.transform(X_test_manual)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


In [ ]:
nb_pred_train = nb_model.predict(X_train_bow)
nb_pred_val = nb_model.predict(X_val_bow)
nb_pred_manual = nb_model.predict(X_test_manual_bow)

registrar_resultado("BoW + Naive Bayes", ENTRENADO_CON,
                     y_train, nb_pred_train,
                     y_val, nb_pred_val,
                     y_test_manual, nb_pred_manual)


## 7. Modelo 2: TF-IDF + Regresión Logística (SGD)

In [ ]:
vectorizer_tfidf = TfidfVectorizer(max_features=100000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_val_tfidf = vectorizer_tfidf.transform(X_val)
X_test_manual_tfidf = vectorizer_tfidf.transform(X_test_manual)

lr_model = SGDClassifier(loss="log_loss", max_iter=20, n_jobs=-1, random_state=RANDOM_STATE)
lr_model.fit(X_train_tfidf, y_train)


In [ ]:
lr_pred_train = lr_model.predict(X_train_tfidf)
lr_pred_val = lr_model.predict(X_val_tfidf)
lr_pred_manual = lr_model.predict(X_test_manual_tfidf)

registrar_resultado("TF-IDF + SGD/LogReg", ENTRENADO_CON,
                     y_train, lr_pred_train,
                     y_val, lr_pred_val,
                     y_test_manual, lr_pred_manual)


## 8. Modelo 3 — "el mejor modelo posible": Word2Vec + clasificador

### 8.1 Entrenar Word2Vec sobre el corpus completo

In [ ]:
tokenized = df["text_clean"].apply(str.split).tolist()

w2v_model = Word2Vec(
    sentences=tokenized,
    vector_size=200,
    window=5,
    min_count=5,
    workers=os.cpu_count(),
    sg=1,            # skipgram
    epochs=5,
)
w2v_model.save("w2v_tweets.model")
print("Vocabulario aprendido:", len(w2v_model.wv))


### 8.2 Vectorizar cada tweet como el promedio de sus embeddings

In [ ]:
def tweet_to_vector(tokens, model):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

X_train_w2v = np.vstack(X_train.apply(str.split).apply(lambda t: tweet_to_vector(t, w2v_model)))
X_val_w2v = np.vstack(X_val.apply(str.split).apply(lambda t: tweet_to_vector(t, w2v_model)))
X_test_manual_w2v = np.vstack(X_test_manual.apply(str.split).apply(lambda t: tweet_to_vector(t, w2v_model)))

clf_w2v = LogisticRegression(max_iter=1000, n_jobs=-1)
clf_w2v.fit(X_train_w2v, y_train)


In [ ]:
w2v_pred_train = clf_w2v.predict(X_train_w2v)
w2v_pred_val = clf_w2v.predict(X_val_w2v)
w2v_pred_manual = clf_w2v.predict(X_test_manual_w2v)

registrar_resultado("Word2Vec + LogReg", ENTRENADO_CON,
                     y_train, w2v_pred_train,
                     y_val, w2v_pred_val,
                     y_test_manual, w2v_pred_manual)


### 8.3 (Opcional, requiere GPU) Red neuronal LSTM

Poné `RUN_LSTM = True` sólo si activaste GPU en el entorno de ejecución y tenés tiempo disponible (ver Nota de la sección 10.3 de la guía: puede tardar bastante incluso con GPU).

In [ ]:
RUN_LSTM = False  # poné True para entrenar la LSTM opcional

if RUN_LSTM:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense

    MAX_LEN = 40
    tok = Tokenizer(num_words=50000)
    tok.fit_on_texts(X_train)

    X_train_seq = pad_sequences(tok.texts_to_sequences(X_train), maxlen=MAX_LEN)
    X_val_seq = pad_sequences(tok.texts_to_sequences(X_val), maxlen=MAX_LEN)
    X_test_manual_seq = pad_sequences(tok.texts_to_sequences(X_test_manual), maxlen=MAX_LEN)

    lstm_model = Sequential([
        Embedding(input_dim=50000, output_dim=200, input_length=MAX_LEN),
        LSTM(64),
        Dense(1, activation="sigmoid"),
    ])
    lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    y_train_bin = (y_train == 4).astype(int)
    y_val_bin = (y_val == 4).astype(int)

    history = lstm_model.fit(X_train_seq, y_train_bin, validation_data=(X_val_seq, y_val_bin),
                              epochs=3, batch_size=1024)

    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="val")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("LSTM - loss por epoch")
    plt.show()

    lstm_pred_train = np.where(lstm_model.predict(X_train_seq, batch_size=1024).ravel() >= 0.5, 4, 0)
    lstm_pred_val = np.where(lstm_model.predict(X_val_seq, batch_size=1024).ravel() >= 0.5, 4, 0)
    lstm_pred_manual = np.where(lstm_model.predict(X_test_manual_seq, batch_size=1024).ravel() >= 0.5, 4, 0)

    registrar_resultado("LSTM (Embedding+LSTM)", ENTRENADO_CON,
                         y_train, lstm_pred_train,
                         y_val, lstm_pred_val,
                         y_test_manual, lstm_pred_manual)


## 9. Comparación de resultados

### 9.1 Tabla comparativa

In [ ]:
resultados_df = pd.DataFrame(resultados).sort_values("accuracy_test_manual", ascending=False).reset_index(drop=True)
resultados_df


### 9.2 Comparación train / validación / test manual (chequeo de overfitting)

Comparar accuracy en train vs. validación vs. test manual sirve para detectar overfitting: si un modelo tiene accuracy mucho más alto en train que en validación/test, probablemente esté memorizando en vez de generalizar.

In [ ]:
comparacion_splits = resultados_df.set_index("modelo")[["accuracy_train", "accuracy_val", "accuracy_test_manual"]]
comparacion_splits.columns = ["Train", "Validación", "Test manual"]

comparacion_splits.plot(kind="bar", figsize=(11, 6))
plt.title("Accuracy por modelo en cada conjunto (train / validación / test manual)")
plt.ylabel("Accuracy")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.legend(title="Conjunto")
plt.tight_layout()
plt.show()


### 9.3 Curva de aprendizaje: ¿ayuda tener más datos de entrenamiento?

Responde una pregunta muy concreta: si el accuracy da más bajo de lo esperado, ¿es porque falta cantidad de datos de entrenamiento, o el techo de performance viene de otro lado (p. ej. la clase neutral ausente en el train)? Para no pagar el costo de reentrenar Word2Vec en cada punto, esta curva se hace sólo con los dos modelos basados en vectorizador + clasificador lineal (BoW+NB y TF-IDF+SGD), que son rápidos de reentrenar desde cero con distintos tamaños de muestra.

In [ ]:
def curva_aprendizaje(nombre_modelo, construir_pipeline, tamanos, X_train_full, y_train_full, X_val, y_val, X_test, y_test):
    filas = []
    for n in tamanos:
        n = min(n, len(X_train_full))
        if n < len(X_train_full):
            X_sub, _, y_sub, _ = train_test_split(
                X_train_full, y_train_full, train_size=n, random_state=RANDOM_STATE, stratify=y_train_full
            )
        else:
            X_sub, y_sub = X_train_full, y_train_full

        vectorizer, modelo = construir_pipeline()
        X_sub_vec = vectorizer.fit_transform(X_sub)
        X_val_vec = vectorizer.transform(X_val)
        X_test_vec = vectorizer.transform(X_test)
        modelo.fit(X_sub_vec, y_sub)

        acc_val = accuracy_score(y_val, modelo.predict(X_val_vec))
        acc_test = accuracy_score(y_test, modelo.predict(X_test_vec))
        filas.append({"modelo": nombre_modelo, "n_train": n, "accuracy_val": acc_val, "accuracy_test_manual": acc_test})
        print(f"{nombre_modelo} | n_train={n:>8} | acc_val={acc_val:.3f} | acc_test_manual={acc_test:.3f}")
    return pd.DataFrame(filas)

def _bow_factory():
    return CountVectorizer(max_features=50000, ngram_range=(1, 1)), MultinomialNB()

def _tfidf_factory():
    return (TfidfVectorizer(max_features=100000, ngram_range=(1, 2), sublinear_tf=True),
            SGDClassifier(loss="log_loss", max_iter=20, n_jobs=-1, random_state=RANDOM_STATE))


In [ ]:
TAMANOS_CURVA = sorted(set(min(t, len(X_train)) for t in [2000, 5000, 20000, 50000, len(X_train)]))

curva_bow = curva_aprendizaje("BoW + Naive Bayes", _bow_factory, TAMANOS_CURVA,
                               X_train, y_train, X_val, y_val, X_test_manual, y_test_manual)
curva_tfidf = curva_aprendizaje("TF-IDF + SGD", _tfidf_factory, TAMANOS_CURVA,
                                 X_train, y_train, X_val, y_val, X_test_manual, y_test_manual)

curva_total = pd.concat([curva_bow, curva_tfidf], ignore_index=True)


In [ ]:
plt.figure(figsize=(10, 6))
for nombre, grupo in curva_total.groupby("modelo"):
    plt.plot(grupo["n_train"], grupo["accuracy_test_manual"], marker="o", label=nombre)
plt.xlabel("Cantidad de tweets de entrenamiento (escala log)")
plt.ylabel("Accuracy en test manual")
plt.title("Curva de aprendizaje: accuracy vs. tamaño del set de entrenamiento")
plt.xscale("log")
plt.legend()
plt.tight_layout()
plt.show()


**Cómo leer este gráfico:** si la curva sigue subiendo con claridad hacia el extremo derecho (dataset completo), es evidencia de que entrenar con más datos ayudaría — apoya la hipótesis de que el accuracy más bajo de lo esperado se debe, al menos en parte, a que `USE_SAMPLE` estaba en `True`. Si la curva se aplana temprano (deja de subir mucho antes de llegar al total), el techo de performance viene de otro lado (por ejemplo, la clase neutral ausente en el train), y agregar más datos del mismo tipo no alcanzaría por sí solo.

### 9.4 Intervalos de confianza (bootstrap) sobre el accuracy

El test manual tiene sólo 498 tweets — un número chico, que hace que el accuracy tenga bastante margen de error de muestreo. Bootstrap permite estimar ese margen: remuestreamos (con reposición) las predicciones muchas veces y medimos cuánto varía el accuracy entre remuestreos.

In [ ]:
def bootstrap_ci_accuracy(y_true, y_pred, n_bootstrap=2000, ci=95, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        accs[i] = accuracy_score(y_true[idx], y_pred[idx])
    alpha = (100 - ci) / 2
    return accs.mean(), np.percentile(accs, alpha), np.percentile(accs, 100 - alpha)

intervalos = []
for nombre, y_pred in predicciones_test_manual.items():
    media, lo, hi = bootstrap_ci_accuracy(y_test_manual.values, y_pred)
    intervalos.append({"modelo": nombre, "accuracy_media_bootstrap": media,
                        "ci_95_lower": lo, "ci_95_upper": hi})

intervalos_df = pd.DataFrame(intervalos).sort_values("accuracy_media_bootstrap", ascending=False).reset_index(drop=True)
intervalos_df


In [ ]:
plt.figure(figsize=(10, 6))
y_pos = np.arange(len(intervalos_df))
plt.errorbar(
    intervalos_df["accuracy_media_bootstrap"], y_pos,
    xerr=[intervalos_df["accuracy_media_bootstrap"] - intervalos_df["ci_95_lower"],
          intervalos_df["ci_95_upper"] - intervalos_df["accuracy_media_bootstrap"]],
    fmt="o", capsize=5, color="steelblue",
)
plt.yticks(y_pos, intervalos_df["modelo"])
plt.xlabel("Accuracy (test manual)")
plt.title("Accuracy con intervalo de confianza del 95% (bootstrap, n=2000 remuestreos)")
plt.tight_layout()
plt.show()


**Cómo interpretar esto:** si los intervalos de dos modelos se superponen bastante, la diferencia entre ellos podría no ser estadísticamente significativa — con sólo 498 tweets de test, una diferencia de accuracy de 1-2 puntos porcentuales (como la que separa a BoW, TF-IDF y Word2Vec en la tabla de la sección 9.1) muy probablemente entra dentro del margen de ruido. Esto es un matiz importante para no sacar conclusiones demasiado categóricas de "el modelo X es mejor que el modelo Y" basándose sólo en el número puntual de accuracy.

### 9.5 Matriz de confusión del mejor modelo

In [ ]:
mejor_modelo_nombre = resultados_df.iloc[0]["modelo"]
mejor_modelo_pred = predicciones_test_manual[mejor_modelo_nombre]
print("Mejor modelo según accuracy en el test manual:", mejor_modelo_nombre)

etiquetas = sorted(y_test_manual.unique())
cm = confusion_matrix(y_test_manual, mejor_modelo_pred, labels=etiquetas)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=etiquetas, yticklabels=etiquetas)
plt.xlabel("Predicho"); plt.ylabel("Real")
plt.title(f"Matriz de confusión - {mejor_modelo_nombre}")
plt.show()


### 9.6 Análisis de errores sistemático

En vez de mirar sólo 5 tweets sueltos al azar, clasificamos TODOS los errores del mejor modelo en categorías heurísticas (negación, contraste/sarcasmo, sentimiento mixto, u "otro"), para ver qué patrón de error es más frecuente. Un mismo tweet puede caer en más de una categoría. Esto es una aproximación simple basada en palabras clave, no un detector real de sarcasmo — se documenta así explícitamente.

In [ ]:
errores = df_test_manual.copy()
errores["prediccion"] = mejor_modelo_pred
errores_mal = errores[errores["target"] != errores["prediccion"]].copy()
print(f"Tweets mal clasificados: {len(errores_mal)} de {len(errores)} ({len(errores_mal)/len(errores)*100:.1f}%)")


In [ ]:
palabras_pos_dict = {"good", "great", "love", "happy", "amazing", "glad", "best", "awesome"}
palabras_neg_dict = {"bad", "hate", "sad", "terrible", "awful", "worst", "annoy", "horrible"}
negadores = {"not", "no", "never", "n't", "dont", "doesnt", "cant", "wont", "didnt", "isnt"}
contraste = {"but", "though", "however", "although", "yet"}

def categorizar_error(texto):
    texto_l = texto.lower()
    palabras = re.findall(r"[a-z']+", texto_l)
    categorias = []
    if any(w in texto_l for w in negadores):
        categorias.append("Negación")
    if any(w in palabras for w in contraste):
        categorias.append("Contraste / posible sarcasmo")
    if any(w in palabras for w in palabras_pos_dict) and any(w in palabras for w in palabras_neg_dict):
        categorias.append("Sentimiento mixto")
    if not categorias:
        categorias.append("Otro / jerga / sin patrón claro")
    return categorias

errores_mal["categorias"] = errores_mal["text"].apply(categorizar_error)

conteo_categorias = Counter(c for cats in errores_mal["categorias"] for c in cats)
conteo_categorias_df = pd.Series(conteo_categorias).sort_values(ascending=False)
conteo_categorias_df


In [ ]:
conteo_categorias_df.plot(kind="barh", figsize=(9, 4), color="indianred")
plt.xlabel("Cantidad de tweets mal clasificados")
plt.title(f"Categorías de error más frecuentes — {mejor_modelo_nombre}")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Un par de ejemplos concretos por categoría, para ilustrar cada patrón de error
for categoria in conteo_categorias_df.index:
    ejemplos = errores_mal[errores_mal["categorias"].apply(lambda cs: categoria in cs)]
    print(f"\n--- {categoria} ({len(ejemplos)} casos) ---")
    for _, fila in ejemplos.head(2).iterrows():
        print(f'  "{fila.text}"  (real={fila.target}, predicho={fila.prediccion})')


## 10. Métrica obligatoria: similitud coseno y PPMI

### 10.1 Similitud coseno entre palabras clave de sentimiento

In [ ]:
palabras_positivas = ["good", "great", "love", "happy", "amazing"]
palabras_negativas = ["bad", "hate", "sad", "terrible", "awful"]

for p1 in palabras_positivas:
    for p2 in palabras_negativas:
        if p1 in w2v_model.wv and p2 in w2v_model.wv:
            sim = w2v_model.wv.similarity(p1, p2)
            print(f"{p1} vs {p2}: {sim:.3f}")


In [ ]:
w2v_model.wv.most_similar("good", topn=10)


### 10.2 Analogías (conecta con la Opción 3 del enunciado)

In [ ]:
# good - happy + sad ≈ ?  (misma lógica que king - man + woman = queen de la clase de Embeddings)
w2v_model.wv.most_similar(positive=["good", "sad"], negative=["happy"], topn=5)


### 10.3 PPMI a partir de una matriz de co-ocurrencias

In [ ]:
from collections import Counter

def construir_coocurrencias(tokenized_texts, window=4, vocab_min=5):
    counts = Counter(t for doc in tokenized_texts for t in doc)
    vocab = [w for w, c in counts.items() if c >= vocab_min]
    idx = {w: i for i, w in enumerate(vocab)}
    co = np.zeros((len(vocab), len(vocab)), dtype=np.float32)
    for doc in tokenized_texts:
        for i, w in enumerate(doc):
            if w not in idx:
                continue
            for j in range(max(0, i - window), min(len(doc), i + window + 1)):
                if i == j or doc[j] not in idx:
                    continue
                co[idx[w], idx[doc[j]]] += 1
    return co, idx

def ppmi(co_matrix):
    total = co_matrix.sum()
    row_sums = co_matrix.sum(axis=1, keepdims=True)
    col_sums = co_matrix.sum(axis=0, keepdims=True)
    expected = row_sums @ col_sums / total
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log2((co_matrix * total) / expected)
    pmi[np.isnan(pmi)] = 0
    return np.maximum(pmi, 0)   # PPMI: negativos -> 0


In [ ]:
muestra_tokens = df["text_clean"].sample(min(100000, len(df)), random_state=RANDOM_STATE).apply(str.split).tolist()
co, idx = construir_coocurrencias(muestra_tokens)
ppmi_matrix = ppmi(co)

if "good" in idx and "great" in idx and "bad" in idx:
    print("PPMI(good, great):", ppmi_matrix[idx["good"], idx["great"]])
    print("PPMI(good, bad):", ppmi_matrix[idx["good"], idx["bad"]])


## 11. Extras opcionales

### 11.1 Wordclouds por sentimiento

In [ ]:
from wordcloud import WordCloud

for clase, nombre in [(4, "positivos"), (0, "negativos")]:
    subset = df[df.target == clase]["text_clean"]
    texto = " ".join(subset.sample(min(20000, len(subset)), random_state=RANDOM_STATE))
    wc = WordCloud(width=800, height=400, background_color="white").generate(texto)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc); plt.axis("off"); plt.title(f"Tweets {nombre}")
    plt.show()


### 11.2 Visualización UMAP de embeddings

In [ ]:
import umap

muestra_umap = df.sample(min(5000, len(df)), random_state=RANDOM_STATE)
vecs = np.vstack(muestra_umap["text_clean"].apply(str.split).apply(lambda t: tweet_to_vector(t, w2v_model)))
emb2d = umap.UMAP(random_state=RANDOM_STATE).fit_transform(vecs)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=emb2d[:, 0], y=emb2d[:, 1], hue=muestra_umap["sentiment"], alpha=0.5)
plt.title("UMAP de tweets coloreado por sentimiento")
plt.show()


### 11.3 Extracción de keywords con BERT (Opción 2 del enunciado)

In [ ]:
from keybert import KeyBERT

kw_model = KeyBERT(model="all-MiniLM-L6-v2")

for clase, nombre in [(4, "positivos"), (0, "negativos")]:
    subset = df[df.target == clase]["text"]
    texto = " ".join(subset.sample(min(2000, len(subset)), random_state=RANDOM_STATE))
    keywords = kw_model.extract_keywords(texto, top_n=10, stop_words="english")
    print(nombre, keywords)


### 11.4 Menú interactivo de analogías (Opción 3 del enunciado)

In [ ]:
def jugar_analogias(a, b, c, model, topn=3):
    """ a es a b como c es a ___ """
    try:
        return model.wv.most_similar(positive=[b, c], negative=[a], topn=topn)
    except KeyError as e:
        return f"Palabra no encontrada en el vocabulario: {e}"

jugar_analogias("bad", "sad", "good", w2v_model)


## 12. Conclusiones

*(Completar esta sección con los resultados reales de tu corrida sobre el dataset completo. Seguí la estructura de la sección 14 de la guía — no dejes afirmaciones sin respaldo en un número de `resultados_df`, `intervalos_df`, `curva_total` o un ejemplo concreto de `errores_mal`.)*

### 12.1 Resumen de resultados
- Mejor modelo: **[completar]**, según **[accuracy / F1 macro / F1 de una clase en particular — elegir y justificar]**.
- Tabla de referencia: `resultados_df` (sección 9.1).
- Chequeo de overfitting: **[completar — ¿algún modelo tiene accuracy mucho más alto en train que en validación/test? Ver el gráfico de la sección 9.2]**.
- Significancia de las diferencias entre modelos: **[completar — según los intervalos de confianza de la sección 9.4, ¿las diferencias de accuracy entre modelos son mayores a lo que el bootstrap sugiere como ruido de muestreo, o se superponen los intervalos?]**.

### 12.2 Justificación de las decisiones tomadas
- Preprocesamiento: **[completar — qué se limpió, por qué, qué impacto tuvo]**.
- Vectorización (BoW / TF-IDF): **[completar — max_features, bigramas, sublinear_tf]**.
- Word2Vec: **[completar — vector_size, window, skipgram vs CBOW, qué se probaría distinto]**.
- Muestreo: **[completar — en qué pasos se usó una muestra y por qué no contradice el requisito de usar los 1.6M de datos]**.
- Tamaño de entrenamiento: **[completar — según la curva de aprendizaje de la sección 9.3, ¿la performance seguía subiendo con más datos, o se había estabilizado? ¿Qué implica esto para la corrida final con USE_SAMPLE=False?]**.

### 12.3 Comparación contra el modelo pre-entrenado (TextBlob)
- **[completar — ¿los modelos propios superaron a TextBlob? ¿por qué? ¿la diferencia es significativa según los intervalos de confianza?]**

### 12.4 Interpretación de la métrica de embeddings (coseno / PPMI)
- **[completar — 2 o 3 ejemplos concretos de la sección 10 y qué dicen sobre la calidad del espacio de embeddings]**

### 12.5 Análisis de errores
- **[completar — según `conteo_categorias_df` (sección 9.6), ¿qué categoría de error es más frecuente (negación, contraste/sarcasmo, sentimiento mixto, u otro)? Ilustrar con 2-3 ejemplos concretos y una hipótesis de por qué falla el modelo en cada caso]**.

### 12.6 Limitaciones
- El set de entrenamiento no tiene clase neutral (2): **[completar]**
- El dataset es de 2009, el lenguaje de Twitter cambió: **[completar]**
- Decisiones de muestreo por tiempo/cómputo: **[completar]**
- La categorización de errores (sección 9.6) es heurística (basada en palabras clave), no un detector real de sarcasmo/negación: **[completar — mencionar como limitación del análisis, no sólo del modelo]**.

### 12.7 Próximos pasos (opcional)
- **[completar — qué probarían con más tiempo/cómputo: BERT/RoBERTa, GloVe/FastText, etc.]**
